In [165]:
import yaml
import json
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import glob
import random
import logging
import shutil

sys.path.append("../../../../donut/src/test/")
from common.config import cfg

In [166]:
logging.basicConfig(
    filename="./logs/YOLODatasetGenerate.log",
    encoding="utf-8",
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
)

In [167]:
class_mapping = {
    "plot": cfg.plot,
    "chart_title": cfg.chart_title,
    "axis_title": cfg.axis_title,
    "tick_label": cfg.tick_label,
    "marker": cfg.marker,
    "visual_element":cfg.visual_element
}

print(class_mapping)

{'plot': 0, 'chart_title': 1, 'axis_title': 2, 'tick_label': 3, 'marker': 4, 'visual_element': 5}


In [168]:
# set the yolo format annotation directory

yolo_format_ds_root_pth_train="../../../data/YOLO-format-annotations/train/labels/"
yolo_format_ds_root_pth_val="../../../data/YOLO-format-annotations/val/labels/"
yolo_format_img_meta_train="../../../data/YOLO-format-annotations/train/images/"
yolo_format_img_meta_val="../../../data/YOLO-format-annotations/val/images/"

In [169]:

image_dir = "../../../data/image_resize/images/"

In [170]:
# list all annotation files paths

annotations_dir="../../../data/image_resize/annotations/"
annotations_files=glob.glob(annotations_dir+"*")
print(len(annotations_files))
print(annotations_files[:5])

60575
['../../../data/image_resize/annotations/e91e28111e86.json', '../../../data/image_resize/annotations/75c0449f6917.json', '../../../data/image_resize/annotations/66dd2a250237.json', '../../../data/image_resize/annotations/58595c30beab.json', '../../../data/image_resize/annotations/497a547454d7.json']


In [171]:
# train test split

random.shuffle(annotations_files)

train_size = int(len(annotations_files) * 0.8)
train_images = annotations_files[:train_size]
val_images = annotations_files[train_size:]

In [172]:
def json_to_yolo(annotation_path, class_mapping=None, orig_image_size=[640, 640]):
    """
    convert json annotations to yolo dataset format

    Params:
        annotation_path: The path to json annotations
        image_path: Images' path
        output_txt_path: Path to store the yolo dataset
        class_mapping: dict, {"class_name": class_id, ...}
    """
    # read JSON
    with open(annotation_path, "r") as f:
        annotation = json.load(f)

    # original image's size
    img_h, img_w = orig_image_size

    object_txt = []

    # main body bbox
    if "plot-bb" in annotation:
        bb = annotation["plot-bb"]
        x0, y0, w, h = bb["x0"], bb["y0"], bb["width"], bb["height"]

        class_id = class_mapping["plot"]

        # YOLO format
        x_center = (x0 + w / 2) / img_w
        y_center = (y0 + h / 2) / img_h
        width = w / img_w
        height = h / img_h

        object_txt.append(
            f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        )

    # text bbox
    if "text" in annotation:
        text = annotation["text"]

        for content in text:
            if content["role"] == "chart_title":
                class_id = class_mapping["chart_title"]

                x_coords = [content["polygon"][f"x{i}"] for i in range(4)]
                y_coords = [content["polygon"][f"y{i}"] for i in range(4)]

                x_min, x_max = min(x_coords), max(x_coords)
                y_min, y_max = min(y_coords), max(y_coords)

                # calculate the x center, y center, width height
                x_center = (x_min + x_max) / 2 / img_w
                y_center = (y_min + y_max) / 2 / img_h
                width = (x_max - x_min) / img_w
                height = (y_max - y_min) / img_h

                object_txt.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                )

            if content["role"] == "axis_title":
                class_id = class_mapping["axis_title"]

                x_coords = [content["polygon"][f"x{i}"] for i in range(4)]
                y_coords = [content["polygon"][f"y{i}"] for i in range(4)]

                x_min, x_max = min(x_coords), max(x_coords)
                y_min, y_max = min(y_coords), max(y_coords)

                x_center = (x_min + x_max) / 2 / img_w
                y_center = (y_min + y_max) / 2 / img_h
                width = (x_max - x_min) / img_w
                height = (y_max - y_min) / img_h

                object_txt.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                )

            if content["role"] == "tick_label":
                class_id = class_mapping["tick_label"]

                x_coords = [content["polygon"][f"x{i}"] for i in range(4)]
                y_coords = [content["polygon"][f"y{i}"] for i in range(4)]

                x_min, x_max = min(x_coords), max(x_coords)
                y_min, y_max = min(y_coords), max(y_coords)

                x_center = (x_min + x_max) / 2 / img_w
                y_center = (y_min + y_max) / 2 / img_h
                width = (x_max - x_min) / img_w
                height = (y_max - y_min) / img_h

                object_txt.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                )

    if "visual-elements" in annotation:
        class_id = class_mapping["visual_element"]

        elements = annotation["visual-elements"]

        if elements["bars"]:
            bars = elements["bars"]

            for bar in bars:
                x0, y0, w, h = bar["x0"], bar["y0"], bar["width"], bar["height"]

                x_center = (x0 + w / 2) / img_w
                y_center = (y0 + h / 2) / img_h
                width = w / img_w
                height = h / img_h

                object_txt.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                )

        else:  # line scatter, dot
            if elements["lines"]:
                ele = elements["lines"]
            elif elements["scatter points"]:
                ele = elements["scatter points"]
            elif elements["boxplots"]:
                ele = elements["boxplots"]
            elif elements["dot points"]:
                ele = elements["dot points"]

            for pts in ele:
                for pt in pts:
                    x_center = pt["x"] / img_w
                    y_center = pt["y"] / img_h
                    # set the bbox size = 6
                    width = 6 / img_w
                    height = 6 / img_h

                    object_txt.append(
                        f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                    )

    return object_txt

In [173]:
# single_annotation_file=train_images[0]
# print(single_annotation_file)
# txt_file_name=single_annotation_file.split("/")[-1].split(".")[0]

# print(txt_file_name)

# object_txt=json_to_yolo(annotation_path=single_annotation_file,class_mapping=class_mapping)
# print(object_txt)

# saved_pth=os.path.join(yolo_format_ds_root_pth_train,f"{txt_file_name}.txt")
# # read into txt file and save the file
# with open(saved_pth, "w") as file:
#     for line in object_txt:
#         file.write(line + "\n")

In [174]:
def yolo_ds_annotation_generate(
    raw_pth: str, location: str, class_mapping: dict, orig_image_size=[640, 640]
):
    """
    generate annotation files in yolo format and save the generated annotation files into target folder

    Params:
        raw_pth: the path to the raw annotation files
        location: the path to the destination
        class_mapping: dict, {"class_name": class_id, ...}
    """

    # get the name of the raw annotation file
    # make sure the name is matched with the image name
    txt_file_name = raw_pth.split("/")[-1].split(".")[0]

    try:
        # Convert the raw content into YOLO format
        object_txt = json_to_yolo(
            annotation_path=raw_pth,
            class_mapping=class_mapping,
            orig_image_size=orig_image_size,
        )

        # Save the result as a .txt file in the target location
        saved_pth = os.path.join(location, f"{txt_file_name}.txt")
        with open(saved_pth, "w") as file:
            for line in object_txt:
                file.write(line + "\n")

    except Exception as e:
        # Log the error without stopping the loop
        logging.error(f"Annotation: Error processing {txt_file_name}: {e}")

    return None
        

In [175]:
# convert and generate all annotation files in yolo format

# train

for pth in train_images:
    yolo_ds_annotation_generate(
        raw_pth=pth, location=yolo_format_ds_root_pth_train, class_mapping=class_mapping
    )

print("all training annotations are prepared")

for pth in val_images:
    yolo_ds_annotation_generate(
        raw_pth=pth, location=yolo_format_ds_root_pth_val, class_mapping=class_mapping
    )

print("all validation annotations are prepared")

all training annotations are prepared
all validation annotations are prepared


In [176]:
# check the num of train and val set

yolo_annotation_train = glob.glob(yolo_format_ds_root_pth_train + "*")

print("the length of train set: ", len(yolo_annotation_train))

yolo_annotation_val = glob.glob(yolo_format_ds_root_pth_val + "*")

print("the length of train set: ", len(yolo_annotation_val))

the length of train set:  48482
the length of train set:  12129


In [177]:
# I hold all raw images in the same place
# I create path links to direct the yolo meta data to the path where the image exists


def get_image_paths(label_dir:str,image_dir:str):
    """
    generate the image paths by the annotation txt file names and the Unified image folder path

    Param:
        label_dir: the path to the yolo format annotation files
    """
    image_paths = []
    for label_file in os.listdir(label_dir):
        if label_file.endswith(".txt"):
            # replace the extension name from .txt to .jpg
            image_name = label_file.replace(".txt", ".jpg")
            image_path = os.path.join(image_dir, image_name)
            if os.path.exists(image_path):
                image_paths.append(image_path)
            else:
                logging.error(
                    f"Image: Error conversion {image_path} not found. Skipping."
                )

    return image_paths



# def write_to_txt(file_path:str, image_paths: list):
#     """
#     write all image_paths into a single txt file

#     Params:
#         file_path: The output file
#         image_paths: The list of all image paths
#     """
#     dwd=os.getcwd().replace("src/test/preprocess","")

#     with open(file_path, 'w') as f:
#         for path in image_paths:
#             new_path=path.replace("../../../", dwd)
#             f.write(new_path + '\n')



# def write_to_txt(file_path:str, image_paths: list):
#     """
#     write all image_paths into a single txt file

#     Params:
#         file_path: The output file
#         image_paths: The list of all image paths
#     """
#     dwd=os.getcwd().replace("src/test/preprocess","")

#     for path in image_paths:
#         file_name=path.split("/")[-1].split(".")[0]
#         new_path=path.replace("../../../", dwd)
#         with open(file_path+f"{file_name}.txt", 'w') as f:
#             new_path=path.replace("../../../", dwd)
#             f.write(new_path + '\n')

In [178]:
def copy_image(image_paths: list, destination_path):
    """
    copy source_path image to destination_path。

    Params:
    source_path (str): Raw image path
    destination_path (str): destination path
    """
    dwd=os.getcwd().replace("src/test/preprocess","")

    for pth in image_paths:
        new_path=pth.replace("../../../", dwd)
        try:
            if not os.path.isfile(new_path):
                raise FileNotFoundError(f"The image does not exist: {new_path}")
            
            if os.path.isdir(destination_path):
                filename = os.path.basename(new_path)
                final_path = os.path.join(destination_path, filename)

            shutil.copy(new_path, final_path)
            logging.info(f"✅: {final_path}")
        except Exception as e:
            logging.error(f"❗ cpoy failed: {e}")

In [179]:
train_images = get_image_paths(yolo_format_ds_root_pth_train,image_dir=image_dir)
val_images = get_image_paths(yolo_format_ds_root_pth_val,image_dir=image_dir)

In [180]:
# check the length

print(len(train_images))
print(len(val_images))
print(train_images[0])
print(val_images[0])

48482
12129
../../../data/image_resize/images/2e83cc736a5a.jpg
../../../data/image_resize/images/b430f34b52e4.jpg


In [181]:
copy_image(image_paths=train_images,destination_path="/Users/yiding/personal_projects/ML/github_repo/donut/data/YOLO-format-annotations/train/images/")

In [182]:
copy_image(image_paths=val_images,destination_path="/Users/yiding/personal_projects/ML/github_repo/donut/data/YOLO-format-annotations/val/images/")

In [183]:
# write_to_txt(file_path=os.path.join(yolo_format_img_meta_train,"train.txt"), image_paths=train_images)
# write_to_txt(file_path=os.path.join(yolo_format_img_meta_val,"val.txt"), image_paths=val_images)

In [184]:
# write_to_txt(file_path=yolo_format_img_meta_train, image_paths=train_images)
# write_to_txt(file_path=yolo_format_img_meta_val, image_paths=val_images)

In [185]:
# ! cat train.txt | xargs ls

# check the path in the train.txt is exact existing